# [2부] 04. 양자화 — 같은 모델을 4분의 1 크기로

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acho98/rebom-lec/blob/main/labs/p2_04_quantization.ipynb)

> 🧭 **스토리** — 팀장의 다음 질문. "사내 서버 GPU가 작은데, 더 큰 모델은 못 올리나?" 같은 모델을 **4비트**로 줄여 메모리·속도·답을 비교하고, T4(15GB)에 **4B 모델**까지 올려 본다.
>
> ⏱ **10분** · 🖥 **GPU 필수**
>
> **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 선택하세요. (GPU가 없으면 작은 설정으로 느리게 돌아갑니다.)
> 💡 **앞 노트북을 열어 둔 상태라면** 그 노트북에서 `런타임 → 런타임 연결 해제 및 삭제`를 먼저 하세요. 무료 코랩은 GPU 세션을 하나만 허용합니다.

**셀 배지**: ⚙️ 준비 · 🔵 따라 실행 · 🟡 바꿔서 실행 · 🤔 생각해보기 · 🟠 심화(시간 남을 때)

## 0. 준비 ⚙️
`bitsandbytes`는 허깅페이스 transformers에 붙는 양자화 라이브러리입니다. **CUDA GPU 전용**이라 CPU 런타임에서는 4비트·8비트 셀을 건너뜁니다(안내 문구가 나옵니다).

In [ ]:
%pip -q install -U "transformers>=4.54" accelerate bitsandbytes

In [ ]:
# ⚙️ GPU 확인 — T4가 잡혔는지, 메모리가 얼마인지
import torch, subprocess
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {name} · 메모리 {total_gb:.1f} GB")
else:
    print("GPU 없음 — CPU로 진행합니다(느립니다). 런타임 유형에서 T4를 선택하면 빨라집니다.")
DEVICE = "cuda" if HAS_GPU else "cpu"
DTYPE = torch.float16 if HAS_GPU else torch.float32   # T4는 bf16을 지원하지 않아 fp16을 씁니다

## 1. 기준선 — fp16 원본 🔵
EXAONE 4.0 1.2B를 fp16으로 올리고 **같은 질문 3개**에 답하게 합니다. 기록할 것은 세 가지입니다.
- **메모리**: 가중치 크기(`get_memory_footprint`)와 생성 중 최대 GPU 사용량(`max_memory_allocated`)
- **속도**: 초당 생성 토큰(토큰/초). GPU는 명령을 받아 두고 나중에 처리하므로, 시간을 재기 전후에 `torch.cuda.synchronize()`로 GPU 작업이 끝나기를 기다립니다. 모델을 올린 뒤 첫 생성은 GPU 준비 시간이 섞여 느리므로 짧게 한 번 돌리고(워밍업) 잽니다.
- **답**: greedy(`do_sample=False`)로 생성해 설정끼리 나란히 놓고 읽습니다.

두 번째 질문은 '인공지능 모델의 가중치를 …'처럼 범위를 적었습니다. '양자화가 무엇인지'만 물으면 모델이 물리학의 양자화로 답합니다.
아래 셀에서 만드는 `load()` · `bench()` · `free()`를 이 노트북의 모든 절이 같이 씁니다.

In [ ]:
import gc, time
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "LGAI-EXAONE/EXAONE-4.0-1.2B"
MAX_NEW = 60 if HAS_GPU else 30                   # CPU는 느리므로 생성 길이를 줄입니다
QUESTIONS = ["국내 출장 숙박비는 어떻게 정산하나요? 두 문장으로 답하세요.",
             "인공지능 모델의 가중치를 16비트에서 4비트로 양자화하는 이유를 한 문장으로 설명하세요.",
             "한빛테크 신입 사원이 첫 출장 전에 확인할 것 세 가지를 짧게 쓰세요."]
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def free():
    """지운 모델이 차지하던 메모리를 돌려받습니다(가비지 컬렉션 → CUDA 캐시 비우기)."""
    gc.collect()
    if HAS_GPU:
        torch.cuda.empty_cache()

def sync():
    """GPU 작업이 끝날 때까지 기다립니다. 시간을 재기 전후에 부릅니다."""
    if HAS_GPU:
        torch.cuda.synchronize()

def load(quant_config=None):
    """같은 모델을 정밀도 설정만 바꿔 로드합니다. quant_config=None 이면 원본(fp16/fp32)."""
    free()
    if HAS_GPU:
        torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE, device_map="auto" if HAS_GPU else None,
                                             quantization_config=quant_config)
    m.eval()
    print(f"로드 {time.perf_counter() - t0:.1f}초 · {m.config._name_or_path} · 가중치 {m.get_memory_footprint() / 1e9:.2f} GB")
    return m

def bench(model, tag, max_new=None, questions=None):
    """질문들을 greedy로 생성하고 (요약 1행, 답 리스트)를 돌려줍니다."""
    questions = questions or QUESTIONS
    answers, n_tok, sec = [], 0, 0.0
    for i, q in enumerate(questions):
        inputs = tokenizer.apply_chat_template([{"role": "user", "content": q}], add_generation_prompt=True,
                                               return_tensors="pt", return_dict=True,
                                               enable_thinking=False).to(model.device)
        if i == 0:                                # 워밍업: 첫 생성은 GPU 준비 시간이 섞여 느리므로 한 번 돌리고 버립니다
            with torch.inference_mode():
                model.generate(**inputs, max_new_tokens=8, do_sample=False)
        sync(); t0 = time.perf_counter()
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=max_new or MAX_NEW, do_sample=False)
        sync(); sec += time.perf_counter() - t0
        new = out[0, inputs["input_ids"].shape[1]:]
        n_tok += len(new)
        answers.append(tokenizer.decode(new, skip_special_tokens=True).strip())
    row = {"설정": tag,
           "가중치 GB": round(model.get_memory_footprint() / 1e9, 2),
           "최대 GPU 메모리 GB": round(torch.cuda.max_memory_allocated() / 1e9, 2) if HAS_GPU else None,
           "토큰/초": round(n_tok / sec, 1),
           "평균 답 길이(토큰)": round(n_tok / len(questions), 1)}
    return row, answers

if "model" in globals():                          # 이 셀을 다시 실행하면 앞에서 올린 모델부터 내립니다
    del model
model = load()                                    # 원본 정밀도
BASE = "fp16 원본" if HAS_GPU else "fp32 원본(CPU)"
row, base_answers = bench(model, BASE)
results, answers = [row], {BASE: base_answers}   # 표 한 줄씩, 설정별 답
print(row)
for q, a in zip(QUESTIONS, base_answers):
    print(f"\nQ. {q}\nA. {a}")

💡 **해석** — 이 행이 **기준선**입니다. T4에서 EXAONE 1.2B fp16의 가중치는 약 2.56GB(1.28B 파라미터 × 2바이트)이고, 생성 중 최대 메모리는 그보다 조금 큽니다(입력·KV 캐시·중간 계산). 토큰/초는 코랩 T4에서 초당 21~27토큰이었습니다(11/16 · 11/17, 다섯 번 실측). 같은 설정도 실행마다 이만큼 달라지므로, 설정끼리는 같은 노트북 안에서 비교하세요. 이 숫자를 기억하고 다음 셀과 비교하세요.

## 2. GPU 메모리 비우기 🔵
무료 T4는 메모리가 약 15GB뿐입니다. 다음 모델을 올리기 전에 **참조 삭제 → 가비지 컬렉션 → CUDA 캐시 비우기**를 순서대로 합니다. 실무에서도 모델을 갈아 끼울 때 똑같이 합니다.

In [ ]:
del model
free()
if HAS_GPU:
    print(f"비운 뒤 GPU 메모리: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
else:
    print("CPU 메모리 해제 완료")

## 3. 4비트(NF4)로 다시 로드 🔵
[허깅페이스 bitsandbytes 문서](https://huggingface.co/docs/transformers/main/en/quantization/bitsandbytes)의 설정 그대로입니다.
- `load_in_4bit=True`: 가중치를 4비트로 저장
- `bnb_4bit_quant_type="nf4"`: 정규분포 모양의 가중치에 맞춘 4비트 형식(NormalFloat4)
- `bnb_4bit_compute_dtype=torch.float16`: 계산할 때는 fp16으로 풀어서 곱합니다. **T4는 bf16을 지원하지 않으므로 fp16**입니다.

In [ ]:
if HAS_GPU:
    bnb4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=torch.float16)
    model = load(bnb4)
    row, answers["4bit NF4"] = bench(model, "4bit NF4")
    results = [r for r in results if r["설정"] != row["설정"]] + [row]   # 이 셀을 다시 실행해도 행이 겹치지 않게
    print(row)
    for q, a in zip(QUESTIONS, answers["4bit NF4"]):
        print(f"\nQ. {q}\nA. {a}")
    del model; free()                              # 다음 실험을 위해 바로 비웁니다
else:
    print("(건너뜀) bitsandbytes 4비트는 CUDA GPU에서만 동작합니다. 런타임 유형을 T4로 바꾸고 다시 실행하세요.")

## 4. 비교 표 🔵
1절과 3절에서 잰 값을 한 표로 모으고, 메모리를 얼마나 줄였는지 · 속도가 어떻게 바뀌었는지 계산합니다. 두 설정의 답도 나란히 놓습니다.

In [ ]:
df = pd.DataFrame(results).set_index("설정")
display(df)
if len(df) > 1:
    base, q4 = df.loc[BASE], df.loc["4bit NF4"]
    print(f"가중치 크기: 원본의 1/{base['가중치 GB'] / q4['가중치 GB']:.1f} (메모리 {1 - q4['가중치 GB'] / base['가중치 GB']:.0%} 절감)")
    print(f"생성 속도: {q4['토큰/초'] / base['토큰/초'] - 1:+.0%}")
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame({"질문": QUESTIONS, **answers}).set_index("질문"))
    print("답의 글자가 달라도 뜻이 같을 수 있습니다. 글자가 같은지는 정확도 지표가 아닙니다.")

💡 **해석** — 표에서 세 가지를 확인하세요.
- **메모리**: 4비트 가중치는 원본의 **1/2~1/3 정도**로 줄어듭니다. 이론상 1/4(16비트 → 4비트)인데 덜 줄어드는 이유는 **임베딩·출력층(`lm_head`)은 양자화하지 않기** 때문입니다. EXAONE 1.2B는 어휘 10만 × 2048차원 임베딩이 약 2억 파라미터로 전체의 6분의 1이라, 작은 모델일수록 이 몫이 큽니다. 모델이 클수록 4배에 가까워집니다.
- **속도**: bitsandbytes 4비트는 **빨라지지 않습니다**. 계산할 때마다 4비트를 fp16으로 풀기 때문에 오히려 느립니다. 코랩 T4에서 이 절의 값은 실행마다 6~36% 느렸고, 5절에서 다시 재면 네 번 모두 26~30% 느렸습니다(11/16 · 11/17). bitsandbytes의 목적은 속도가 아니라 **메모리 절약**입니다.
- **답**: greedy인데도 답이 원본과 글자 단위로 달라질 수 있습니다. 가중치가 조금 바뀌면 어느 시점에서 1등 토큰이 바뀌고, 그 뒤 문장 전체가 갈라집니다. 뜻이 같은지 직접 읽어 보세요. 글자가 같은지는 정확도 지표가 아닙니다.

## 5. 심화 🟠 (시간 남을 때) — 8비트까지 같은 조건으로 비교
아래 셀은 원본 · 8비트(`load_in_8bit=True`, LLM.int8) · 4비트를 **같은 질문, 같은 최대 생성 길이**로 다시 잽니다. 1 · 3절과 같은 `load()` · `bench()`를 씁니다.
1. 메모리가 4비트와 원본 사이에 오는지, 속도는 어떻게 되는지 보세요.
2. 질문 3개는 답이 30토큰 안팎에서 끝나서 `MAX_NEW_TRY`를 늘려도 생성 길이가 거의 같습니다. 그래서 답이 길게 나오는 질문을 하나 더했습니다.

In [ ]:
MAX_NEW_TRY = 200                                       # ← 60 으로 줄여 1 · 3절과 같은 길이로도 재 보세요
TRY_QUESTIONS = QUESTIONS + ["한빛테크 신입 사원을 위한 국내 출장 준비 체크리스트를 10개 항목으로 자세히 쓰세요."]   # 마지막 질문은 답이 깁니다
TRY_CONFIGS = {"fp16 원본": None,
               "8bit": BitsAndBytesConfig(load_in_8bit=True),
               "4bit NF4": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                              bnb_4bit_compute_dtype=torch.float16)}
if HAS_GPU:
    try_rows, try_answers = [], {}
    for tag, cfg in TRY_CONFIGS.items():
        model = load(cfg)
        row, try_answers[tag] = bench(model, tag, max_new=MAX_NEW_TRY, questions=TRY_QUESTIONS)
        try_rows.append(row)
        del model; free()
    try_df = pd.DataFrame(try_rows).set_index("설정")
    display(try_df)
    for tag in ["8bit", "4bit NF4"]:
        print(f"{tag}: 가중치 {1 - try_df.loc[tag, '가중치 GB'] / try_df.loc['fp16 원본', '가중치 GB']:.0%} 절감 · "
              f"생성 속도 {try_df.loc[tag, '토큰/초'] / try_df.loc['fp16 원본', '토큰/초'] - 1:+.0%}")
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame({"질문": TRY_QUESTIONS, **try_answers}).set_index("질문"))
else:
    print("(건너뜀) bitsandbytes 8비트도 CUDA GPU 전용입니다.")

💡 **해석** — 코랩 T4 실측입니다(최대 200토큰. 11/16은 질문 3개로 한 번, 11/17은 체크리스트 질문을 더한 4개로 세 번).

| 설정 | 가중치 GB | 최대 GPU 메모리 GB | 토큰/초 | 원본 대비 |
|---|---|---|---|---|
| fp16 원본 | 2.56 | 2.57~2.58 | 24.4~26.3 | 기준 |
| 8bit | 1.49 | 1.52 | 7.8~8.3 | 가중치 42% 절감 · 속도 -67~-69% |
| 4bit NF4 | 0.95 | 0.99~1.05 | 17.5~18.9 | 가중치 63% 절감 · 속도 -26~-30% |

- **메모리**는 예상대로 원본 > 8비트 > 4비트입니다.
- **8비트가 가장 느립니다.** bitsandbytes의 8비트(LLM.int8)는 값이 큰 이상치 열을 떼어 fp16으로 곱하고, 나머지만 INT8로 곱해 합칩니다. 정확도를 지키는 대신 이상치 분리 · 활성값 양자화 · 역양자화 같은 단계가 더 붙어서, 작은 모델로 한 건씩 생성할 때 느립니다.
- 4비트의 속도 차이는 4절에서 실행마다 -6~-36%로 흔들렸고, 이 절에서는 네 번 모두 -26~-30%였습니다. 한 번 잰 값보다 여러 번 잰 범위로 판단하세요.
- 8비트 모델은 두 번째 질문에 '계산 정밀도가 감소하여 연산 속도가 빨라지지만'이라고 답했습니다. 위 표에서는 8비트가 가장 느립니다. 모델의 설명보다 실측을 믿으세요.

## 6. T4에 4B 모델 올리기 🔵 — 사전 양자화 모델
4비트로 줄이면 **같은 GPU에 더 큰 모델**이 들어갑니다. `unsloth/Qwen3-4B-unsloth-bnb-4bit`는 Qwen3-4B(Apache 2.0)를 **미리 4비트로 저장해 둔** 모델입니다. 로그인·동의 없이 받을 수 있고 다운로드는 약 3.6GB입니다(1~2분).
저장된 설정의 계산 dtype이 bf16이라 T4용으로 **fp16으로 바꿔서** 로드합니다. 시간이 부족하면 이 절은 건너뛰고 7절을 읽으세요.

In [ ]:
from transformers import AutoConfig
Q4B = "unsloth/Qwen3-4B-unsloth-bnb-4bit"
if HAS_GPU:
    cfg = AutoConfig.from_pretrained(Q4B)
    cfg.quantization_config["bnb_4bit_compute_dtype"] = "float16"     # 저장값 bfloat16 → T4는 fp16
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    tok4 = AutoTokenizer.from_pretrained(Q4B)
    model4 = AutoModelForCausalLM.from_pretrained(Q4B, config=cfg, dtype=torch.float16, device_map="auto")
    print(f"로드 {time.perf_counter()-t0:.0f}초 · 파라미터 {sum(p.numel() for p in model4.parameters())/1e9:.2f}B(4비트 묶음 기준)"
          f" · 가중치 {model4.get_memory_footprint()/1e9:.2f} GB")

    msgs = [{"role": "user", "content": QUESTIONS[0]}]
    inputs = tok4.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True,
                                      enable_thinking=False).to(model4.device)   # Qwen3도 사고 모드를 끕니다
    with torch.inference_mode():
        model4.generate(**inputs, max_new_tokens=8, do_sample=False)   # 워밍업(1절과 같은 방식)
    sync(); t0 = time.perf_counter()
    with torch.inference_mode():
        out = model4.generate(**inputs, max_new_tokens=MAX_NEW, do_sample=False)
    sync(); sec = time.perf_counter() - t0
    new = out[0, inputs["input_ids"].shape[1]:]
    print(f"생성 {len(new)/sec:.1f} 토큰/초 · 최대 GPU 메모리 {torch.cuda.max_memory_allocated()/1e9:.2f} GB"
          f" / 전체 {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print("\nQ.", QUESTIONS[0], "\nA.", tok4.decode(new, skip_special_tokens=True).strip())
else:
    print("(건너뜀) 사전 양자화 bnb-4bit 모델도 CUDA GPU가 필요합니다.")

💡 **해석** — Qwen3-4B를 fp16으로 올리면 가중치만 약 8GB입니다. 4비트로는 **3.5GB 안팎**(코랩 T4 실측 3.50GB)이라 T4에 넉넉히 들어가고, 1.2B 원본(2.56GB)과 비슷한 메모리로 **파라미터가 3배 넘는 모델**을 쓰는 셈입니다. 대신 생성은 초당 10토큰 안팎(코랩 T4 9.6~11.2)으로 1.2B 원본의 절반 안팎입니다. 답의 품질이 1.2B와 어떻게 다른지 1절의 첫 번째 답과 비교해 보세요.

## 7. GGUF — 노트북 05로 가는 다리 📖
bitsandbytes는 **파이썬·transformers·CUDA 안에서** 쓰는 양자화입니다. 반면 **GGUF**는 llama.cpp 계열 엔진(Ollama, LM Studio 등)이 쓰는 **파일 형식**입니다. 양자화된 가중치와 토크나이저·채팅 템플릿을 파일 하나에 담아서 **CPU·맥·윈도우에서도** 돌릴 수 있습니다.
- 이름의 `Q4_K_M`은 "4비트, K-quant 방식, 중간(M) 품질"이라는 뜻입니다. EXAONE 4.0 1.2B의 `Q4_K_M` GGUF는 **812MB**입니다(fp16 2.56GB의 약 1/3).
- 노트북 05에서 이 파일을 Ollama로 받아 **OpenAI와 똑같은 코드**로 부릅니다.
- AWQ·GPTQ(사전 양자화 형식)와 vLLM·`transformers serve`(서빙 엔진)는 설치가 무거워 장표에서 소개만 합니다.

**공식 문서에 나온 숫자**

| 비교 | 결과 | 출처 |
|---|---|---|
| Qwen3-8B **AWQ 4비트** vs **bf16** (MMLU-Redux) | 86.4 vs 87.5 (1.1점 하락) | Qwen3-8B-AWQ 모델 카드 |
| bitsandbytes **4비트** 메모리 | 원본 대비 약 **4배 절감** | HF bitsandbytes 문서 |
| EXAONE 4.0 1.2B 파일 크기 | fp16 2.56GB → GGUF Q4_K_M 812MB | 각 모델 저장소 |

### 🤔 생각해보기

**4비트 4B 모델과 fp16 1.2B 모델, 어느 쪽이 나은가?** 무엇으로 판단해야 할까요?

<details><summary><b>🤔 생각해보기 — 해설 보기(클릭)</b></summary>

정답은 하나가 아닙니다. **같은 메모리 예산**에서 무엇을 사느냐의 문제입니다.
- **품질**: 일반적으로 "큰 모델을 양자화"하는 쪽이 "작은 모델 원본"보다 성능이 좋습니다(Qwen3-8B는 4비트로 줄여도 MMLU-Redux가 1.1점만 떨어짐). 하지만 **우리 업무 질문**으로 직접 재야 합니다. 규정 질문 20개를 만들어 두 모델의 정답률을 비교하세요.
- **속도**: 4B는 계산량이 3배 이상이고 bitsandbytes는 풀기 비용까지 있어 느립니다. 응답 시간 요구(예: 2초 안)가 있으면 1.2B가 유리할 수 있습니다.
- **라이선스**: EXAONE 1.2B는 비상업(NC) 라이선스라 회사 서비스에는 쓸 수 없습니다. Qwen3는 Apache 2.0입니다. **실무에서는 이것이 먼저 걸러지는 조건**입니다.
- **한국어**: 한국어 토큰 효율(1부 노트북 02)과 한국어 답의 자연스러움도 판단 기준입니다.

</details>

---
### ✅ 이 노트북에서 가져갈 것
- 양자화는 가중치의 비트 수를 줄여 메모리를 아낀다. 4비트 NF4 + fp16 계산이 T4의 기본 조합이다.
- 작은 모델은 임베딩·출력층 몫이 커서 4배보다 덜 줄어든다. bitsandbytes는 속도가 아니라 메모리를 위한 도구다.
- greedy라도 양자화하면 답이 글자 단위로 달라질 수 있다. 품질은 업무 질문으로 직접 잰다.
- 4비트로 줄이면 T4에 4B 모델이 들어간다. GGUF는 같은 아이디어를 파일 형식으로 담아 CPU에서도 돌린다.
- 다음 노트북으로 가기 전 `런타임 → 런타임 연결 해제 및 삭제`로 GPU를 비우세요.

> 다음 노트북: **p2_05_openai_compatible_serving — 같은 코드로 로컬 모델과 OpenAI를 번갈아 부르기**